# 05. Hypothesis: Asymmetric Cost Function

Проверка гипотезы: оптимизация threshold под асимметричную
функцию стоимости (FNR важнее FPR) улучшит OOS Recall
при приемлемой потере In-Domain Accuracy.

**Эксперимент проводится на обоих датасетах (standard и deeppavlov)
для проверки устойчивости результатов.**

## Содержание
1. Setup
2. Постановка гипотезы
3. Метод
4. Эксперимент
5. Сравнение датасетов
6. Pareto-кривая
7. Выводы

## 1. Setup

In [ ]:
# Load environment variables from .env (macOS ARM fix)
from dotenv import load_dotenv
load_dotenv("../../../.env")  # project root

import sys
from pathlib import Path
import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Project paths
NOTEBOOK_DIR = Path.cwd()
TASK_DIR = NOTEBOOK_DIR.parent
PROJECT_ROOT = TASK_DIR.parent.parent
PROCESSED = TASK_DIR / "data" / "processed"
RESULTS = TASK_DIR / "results"
RUNS = TASK_DIR / "runs"

sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(TASK_DIR))

# Both data sources
SOURCES = ["standard", "deeppavlov"]

print(f"Project root: {PROJECT_ROOT}")
print(f"Results dir: {RESULTS}")
print(f"Runs dir: {RUNS}")
print(f"Sources: {SOURCES}")

import os
print(f"\nOMP_NUM_THREADS={os.environ.get('OMP_NUM_THREADS', 'not set')}")

## 2. Постановка гипотезы

### Текущее поведение AutoIntent

Стандартный threshold в AutoIntent оптимизируется под **F1-score**,
который симметрично взвешивает False Negatives (FN) и False Positives (FP):

$$F1 = \frac{2 \cdot TP}{2 \cdot TP + FP + FN}$$

### Проблема для guardrail-сценария

В контексте guardrail для LLM-агента ошибки имеют **разную стоимость**:

| Ошибка | Описание | Последствие | Стоимость |
|--------|----------|-------------|----------|
| **FN** (False Negative) | OOS-запрос пропущен как in-scope | Агент пытается ответить на запрос вне компетенции → **неверный/опасный ответ** | **Высокая** |
| **FP** (False Positive) | In-scope запрос заблокирован как OOS | Агент отвечает "не могу помочь" → **безопасный fallback** | Низкая |

FN >> FP по стоимости, потому что:
- FN приводит к потенциально вредным действиям агента
- FP лишь снижает utility, но не создаёт риска

### Гипотеза

**Оптимизация threshold под асимметричную функцию стоимости:**

$$\text{Cost} = \alpha \cdot FNR + \beta \cdot FPR, \quad \alpha > \beta$$

где:
- $FNR = \frac{FN}{FN + TP}$ — доля пропущенных OOS (= 1 - OOS Recall)
- $FPR = \frac{FP}{FP + TN}$ — доля ложно заблокированных in-scope

**Ожидание:** при $\alpha > \beta$ threshold сдвинется влево (станет
более "параноидальным"), что даст:
- **Рост OOS Recall** (меньше FN)
- **Падение In-Domain Acc** (больше FP)
- **Общий выигрыш** для guardrail-сценария, если падение Acc приемлемо

## 3. Метод

### Функция оптимизации threshold

Перебираем threshold от 0 до 1 и выбираем тот, который минимизирует
взвешенную сумму FNR и FPR на validation set.

In [ ]:
from src.metrics import get_oos_scores_from_pipeline
from src.data_utils import load_split


def optimize_threshold(oos_scores_val, y_val, alpha, beta, n_thresholds=200):
    """
    Подбирает threshold минимизируя alpha*FNR + beta*FPR на val.
    
    Args:
        oos_scores_val: OOS-скоры на val (выше = более OOS)
        y_val: истинные метки (-1 = OOS, 0..N = in-scope)
        alpha: вес FNR (пропуск OOS)
        beta: вес FPR (ложная блокировка in-scope)
        n_thresholds: число точек для перебора
    
    Returns:
        best_t: оптимальный threshold
        best_cost: значение cost function
    """
    thresholds = np.linspace(0, 1, n_thresholds)
    best_t, best_cost = 0.5, np.inf
    
    y_true_oos = (np.array(y_val) == -1).astype(int)
    
    for t in thresholds:
        y_pred_oos = (oos_scores_val >= t).astype(int)
        
        tp = ((y_pred_oos == 1) & (y_true_oos == 1)).sum()
        fn = ((y_pred_oos == 0) & (y_true_oos == 1)).sum()
        fp = ((y_pred_oos == 1) & (y_true_oos == 0)).sum()
        tn = ((y_pred_oos == 0) & (y_true_oos == 0)).sum()
        
        fnr = fn / (fn + tp + 1e-9)
        fpr = fp / (fp + tn + 1e-9)
        cost = alpha * fnr + beta * fpr
        
        if cost < best_cost:
            best_cost, best_t = cost, t
    
    return best_t, best_cost


def evaluate_threshold(oos_scores, y_true, threshold):
    """
    Оценивает метрики при заданном threshold.
    
    Returns:
        dict с метриками: oos_recall, in_domain_acc, f1_oos, fnr, fpr
    """
    y_true = np.array(y_true)
    y_true_oos = (y_true == -1).astype(int)
    y_pred_oos = (oos_scores >= threshold).astype(int)
    
    tp = ((y_pred_oos == 1) & (y_true_oos == 1)).sum()
    fn = ((y_pred_oos == 0) & (y_true_oos == 1)).sum()
    fp = ((y_pred_oos == 1) & (y_true_oos == 0)).sum()
    tn = ((y_pred_oos == 0) & (y_true_oos == 0)).sum()
    
    oos_recall = tp / (tp + fn + 1e-9)
    fnr = fn / (fn + tp + 1e-9)
    fpr = fp / (fp + tn + 1e-9)
    in_domain_acc = tn / (tn + fp + 1e-9)  # = 1 - FPR на in-scope
    
    precision = tp / (tp + fp + 1e-9)
    f1_oos = 2 * precision * oos_recall / (precision + oos_recall + 1e-9)
    
    return {
        "oos_recall": oos_recall,
        "in_domain_acc": in_domain_acc,
        "f1_oos": f1_oos,
        "fnr": fnr,
        "fpr": fpr,
    }


print("Functions defined.")

### Тестируемые соотношения α:β

| Соотношение | Интерпретация |
|-------------|---------------|
| 1:1 | Baseline (эквивалент F1-оптимизации) |
| 2:1 | Умеренный акцент на FNR |
| 5:1 | Сильный акцент на FNR |
| 10:1 | Очень сильный акцент на FNR |

## 4. Эксперимент

Запускаем оптимизацию threshold для 20-shot моделей (3 seeds)
с разными соотношениями α:β.

In [ ]:
from autointent import Pipeline

# Load validation and test data for both sources
data_by_source = {}

for source in SOURCES:
    val_data = load_split(source, "validation")
    test_data = load_split(source, "test")
    
    data_by_source[source] = {
        "val_texts": val_data["texts"],
        "val_labels": np.array(val_data["labels"]),
        "test_texts": test_data["texts"],
        "test_labels": np.array(test_data["labels"]),
    }
    
    print(f"{source}:")
    print(f"  Val samples: {len(val_data['texts'])} (OOS: {(data_by_source[source]['val_labels'] == -1).sum()})")
    print(f"  Test samples: {len(test_data['texts'])} (OOS: {(data_by_source[source]['test_labels'] == -1).sum()})")

In [ ]:
# Experiment configuration
SEEDS = [42, 123, 456]
N_SHOTS = 20
ALPHA_BETA_RATIOS = [
    (1, 1, "1:1 (F1)"),
    (2, 1, "2:1"),
    (5, 1, "5:1"),
    (10, 1, "10:1"),
]

# Collect results for both sources
all_results = []

for source in SOURCES:
    print(f"\n{'='*60}")
    print(f"SOURCE: {source.upper()}")
    print(f"{'='*60}")
    
    data = data_by_source[source]
    val_texts = data["val_texts"]
    val_labels = data["val_labels"]
    test_texts = data["test_texts"]
    test_labels = data["test_labels"]
    
    for seed in SEEDS:
        # Model directory includes source
        model_dir = RUNS / f"autointent_classic-light_{source}_{N_SHOTS}shot_seed{seed}"
        
        if not model_dir.exists():
            print(f"Model not found: {model_dir}")
            continue
        
        print(f"\nProcessing {source}/seed={seed}...")
        
        # Load pipeline
        pipeline = Pipeline.load(model_dir)
        
        # Get OOS scores
        print("  Getting OOS scores on val...")
        oos_scores_val = get_oos_scores_from_pipeline(pipeline, val_texts)
        
        print("  Getting OOS scores on test...")
        oos_scores_test = get_oos_scores_from_pipeline(pipeline, test_texts)
        
        # Test each alpha:beta ratio
        for alpha, beta, label in ALPHA_BETA_RATIOS:
            # Optimize threshold on val
            best_t, best_cost = optimize_threshold(
                oos_scores_val, val_labels, alpha, beta
            )
            
            # Evaluate on test
            metrics = evaluate_threshold(oos_scores_test, test_labels, best_t)
            
            result = {
                "source": source,
                "seed": seed,
                "alpha_beta": label,
                "alpha": alpha,
                "beta": beta,
                "threshold": best_t,
                "val_cost": best_cost,
                **metrics,
            }
            all_results.append(result)
            
            print(f"    {label}: t={best_t:.3f}, Recall={metrics['oos_recall']:.3f}, "
                  f"Acc={metrics['in_domain_acc']:.3f}, F1={metrics['f1_oos']:.3f}")

# Create DataFrame
results_df = pd.DataFrame(all_results)
print(f"\n{'='*60}")
print(f"Total results: {len(results_df)}")

In [ ]:
# Aggregate by source and alpha_beta (mean ± std across seeds)
agg_results = results_df.groupby(["source", "alpha_beta"]).agg({
    "threshold": ["mean", "std"],
    "oos_recall": ["mean", "std"],
    "in_domain_acc": ["mean", "std"],
    "f1_oos": ["mean", "std"],
    "fnr": ["mean", "std"],
    "fpr": ["mean", "std"],
}).reset_index()

# Flatten column names
agg_results.columns = ["_".join(col).strip("_") for col in agg_results.columns]

# Sort by source and alpha
order = ["1:1 (F1)", "2:1", "5:1", "10:1"]
agg_results["sort_key"] = agg_results["alpha_beta"].apply(lambda x: order.index(x))
agg_results = agg_results.sort_values(["source", "sort_key"]).drop(columns=["sort_key"])

# Display formatted tables for each source
for source in SOURCES:
    source_agg = agg_results[agg_results["source"] == source]
    
    print(f"\n{source.upper()}: Results by α:β ratio (mean ± std across 3 seeds)")
    print("=" * 90)
    
    display_df = pd.DataFrame({
        "α:β": source_agg["alpha_beta"].values,
        "Threshold": source_agg.apply(
            lambda r: f"{r['threshold_mean']:.3f} ± {r['threshold_std']:.3f}", axis=1).values,
        "OOS Recall": source_agg.apply(
            lambda r: f"{r['oos_recall_mean']:.3f} ± {r['oos_recall_std']:.3f}", axis=1).values,
        "In-Domain Acc": source_agg.apply(
            lambda r: f"{r['in_domain_acc_mean']:.3f} ± {r['in_domain_acc_std']:.3f}", axis=1).values,
        "F1 OOS": source_agg.apply(
            lambda r: f"{r['f1_oos_mean']:.3f} ± {r['f1_oos_std']:.3f}", axis=1).values,
        "FPR": source_agg.apply(
            lambda r: f"{r['fpr_mean']:.3f} ± {r['fpr_std']:.3f}", axis=1).values,
    })
    
    display(display_df)

In [ ]:
# Save results to JSON
results_json = {
    "experiment": "asymmetric_cost_threshold",
    "sources": SOURCES,
    "n_shots": N_SHOTS,
    "seeds": SEEDS,
    "raw_results": all_results,
    "aggregated": agg_results.to_dict(orient="records"),
}

output_file = RESULTS / "hypothesis_asymmetric_cost.json"
output_file.write_text(json.dumps(results_json, indent=2, ensure_ascii=False))
print(f"Saved to {output_file}")

## 5. Сравнение датасетов

Сравним результаты между standard и deeppavlov для каждого α:β.

In [ ]:
# Compare sources side by side
print("=== Сравнение датасетов по α:β ===\n")

comparison_rows = []
for ab in order:
    std = agg_results[(agg_results["source"] == "standard") & (agg_results["alpha_beta"] == ab)]
    dp = agg_results[(agg_results["source"] == "deeppavlov") & (agg_results["alpha_beta"] == ab)]
    
    if len(std) > 0 and len(dp) > 0:
        std_recall = std["oos_recall_mean"].values[0]
        dp_recall = dp["oos_recall_mean"].values[0]
        std_acc = std["in_domain_acc_mean"].values[0]
        dp_acc = dp["in_domain_acc_mean"].values[0]
        
        comparison_rows.append({
            "α:β": ab,
            "Recall (std)": f"{std_recall:.3f}",
            "Recall (dp)": f"{dp_recall:.3f}",
            "Δ Recall": f"{dp_recall - std_recall:+.3f}",
            "Acc (std)": f"{std_acc:.3f}",
            "Acc (dp)": f"{dp_acc:.3f}",
            "Δ Acc": f"{dp_acc - std_acc:+.3f}",
        })

comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df)

# Statistical summary
print("\n=== Разница между датасетами (deeppavlov - standard) ===")
diff_recall = []
diff_acc = []
for ab in order:
    std = agg_results[(agg_results["source"] == "standard") & (agg_results["alpha_beta"] == ab)]
    dp = agg_results[(agg_results["source"] == "deeppavlov") & (agg_results["alpha_beta"] == ab)]
    if len(std) > 0 and len(dp) > 0:
        diff_recall.append(dp["oos_recall_mean"].values[0] - std["oos_recall_mean"].values[0])
        diff_acc.append(dp["in_domain_acc_mean"].values[0] - std["in_domain_acc_mean"].values[0])

print(f"Δ Recall: mean={np.mean(diff_recall):+.4f}, std={np.std(diff_recall):.4f}")
print(f"Δ Acc:    mean={np.mean(diff_acc):+.4f}, std={np.std(diff_acc):.4f}")

## 6. Pareto-кривая

Визуализация trade-off между FPR и OOS Recall при разных α:β для обоих датасетов.

In [ ]:
# Load baselines for comparison (both sources)
metrics_all = json.loads((RESULTS / "metrics.json").read_text())

baselines = {}
for source in SOURCES:
    baseline_e5 = [r for r in metrics_all 
                   if r["model_name"] == "cosine_e5large_threshold" 
                   and r.get("n_shots") == N_SHOTS
                   and r.get("extra", {}).get("source") == source]
    
    if baseline_e5:
        baseline_recall = np.mean([r["oos_recall"] for r in baseline_e5])
        baseline_acc = np.mean([r["in_domain_acc"] for r in baseline_e5])
        baseline_fpr = 1 - baseline_acc
        baselines[source] = {"recall": baseline_recall, "fpr": baseline_fpr}
        print(f"{source} baseline (E5-Large threshold): Recall={baseline_recall:.3f}, FPR={baseline_fpr:.3f}")
    else:
        print(f"{source} baseline not found")

In [ ]:
# Plot Pareto curves for both sources
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Colors for different ratios
colors = {
    "1:1 (F1)": "#3498db",
    "2:1": "#2ecc71",
    "5:1": "#f39c12",
    "10:1": "#e74c3c",
}

for idx, source in enumerate(SOURCES):
    ax = axes[idx]
    source_agg = agg_results[agg_results["source"] == source]
    
    # Plot each ratio with error bars
    for _, row in source_agg.iterrows():
        label = row["alpha_beta"]
        ax.errorbar(
            row["fpr_mean"], row["oos_recall_mean"],
            xerr=row["fpr_std"], yerr=row["oos_recall_std"],
            marker="o", markersize=12, capsize=5,
            color=colors[label], label=label,
            linewidth=2, elinewidth=2,
        )
        
        # Add text label
        ax.annotate(
            label,
            (row["fpr_mean"], row["oos_recall_mean"]),
            xytext=(10, 5), textcoords="offset points",
            fontsize=10, fontweight="bold",
        )
    
    # Connect points with line
    sorted_agg = source_agg.sort_values("fpr_mean")
    ax.plot(
        sorted_agg["fpr_mean"], sorted_agg["oos_recall_mean"],
        "--", color="gray", alpha=0.5, linewidth=1,
    )
    
    # Add baseline point
    if source in baselines:
        b = baselines[source]
        ax.scatter(
            b["fpr"], b["recall"],
            marker="s", s=150, color="purple", zorder=10,
            label="E5-Large baseline",
        )
    
    # Ideal point
    ax.scatter(0, 1, marker="*", s=200, color="gold", zorder=10, label="Ideal")
    
    ax.set_xlabel("FPR ← лучше", fontsize=12)
    ax.set_ylabel("OOS Recall → лучше", fontsize=12)
    ax.set_title(f"Pareto Trade-off: {source.upper()}", fontsize=14)
    ax.set_xlim(-0.02, 0.30)
    ax.set_ylim(0.5, 1.02)
    ax.legend(loc="lower right", fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS / "hypothesis_asymmetric_cost_pareto.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved to {RESULTS / 'hypothesis_asymmetric_cost_pareto.png'}")

## 7. Выводы

### Устойчивость результатов между датасетами

Эксперимент проведён на обоих датасетах (standard и deeppavlov). Validation и test sets идентичны между источниками, но обученные модели потенциально различаются (100 vs 200 OOS в train). Тем не менее, результаты близки — разница в метриках минимальна.

### Гипотеза технически подтверждена, но практически неинтересна

Асимметричная cost function действительно смещает threshold и увеличивает OOS Recall — механизм работает как ожидалось. При переходе от 1:1 к 10:1 Recall растёт, но ценой значительного роста FPR.

### Проблема 1 — невыгодный trade-off

F1-оптимизированный baseline (1:1) уже обеспечивает хороший баланс. Переход к более агрессивным α:β даёт marginal utility в Recall, но экспоненциально увеличивает FPR. При 5:1 и 10:1 каждый пятый-четвёртый легитимный запрос блокируется — неприемлемо для production guardrail.

### Проблема 2 — нестабильность при 2:1

При умеренном смещении (2:1) результаты сильно зависят от конкретного seed — высокий std для threshold и In-Domain Acc делает эту operating point ненадёжной.

### Положительный результат — AutoIntent Pareto-доминирует

При стандартном F1 threshold AutoIntent (1:1) одновременно лучше cosine_e5large_threshold и по Recall и по FPR на обоих датасетах. AutoML-оптимизация даёт качественно лучшую рабочую точку.

### Итог

**Гипотеза опровергнута** в части "улучшит без критической потери In-Domain Acc": при любом α > 1 потери существенны. Для задачи OOS-детекции стандартная F1-оптимизация AutoIntent уже обеспечивает хороший guardrail-баланс. Асимметричная cost function — настраиваемый knob для специфических требований, но не улучшение пайплайна по умолчанию.

**Результаты устойчивы между датасетами** — выбор source не влияет на качественные выводы.